# 1. Título: 

### Dados Consolidados de Movimentação de Gás Natural em Gasodutos de Transporte

# 2. Membros (nome e número de matrícula): 

### Igor Braga de Lima - 2021019351
### Arthur Felipe Reis Souza - 2021013884
### Raphael Henrique Braga Leivas - 2020028101

# 3. Descrição dos dados (qual a URL? qual o domínio? como os dados foram processados?)

URL: https://dados.gov.br/dados/conjuntos-dados/dados-consolidados-de-movimentacao-de-gas-natural-em-gasodutos-de-transporte
Domínio: Agência Nacional de Petróleo, Gás Natural e Biocombustíveis

Inicialmente completou-se as datas faltantes, seguindo a ordem cronologica da amostragem. Em seguida 

# 4. Diagrama ER

![title](img/diagrama-er.png)

# 5. Diagrama relacional

![title](img/modelo-relacional.png)

# 6. Consultas

In [46]:
import pandas as pd
import sqlite3

In [47]:
# Conectando ao banco de dados SQLite
conn = sqlite3.connect('gas_data.db')

# Listando as tabelas no banco de dados
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql_query(tables_query, conn)

print("As tabelas disponíveis no banco de dados são:")
print(tables_df)

# Loop em todas as tabelas para exibir as primeiras linhas
for table_name in tables_df['name']:
    print(f"\n📄 Primeiras linhas da tabela: {table_name}")
    df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5;", conn)
    print(df)

As tabelas disponíveis no banco de dados são:
                    name
0               Operador
1             Carregador
2  Instalacao_Transporte
3    Instalacao_Gasoduto
4               Contrato
5                Medicao
6               Variavel

📄 Primeiras linhas da tabela: Operador
   codigo_operador                                      nome_operador
0                5  Transportadora Brasileira Gasoduto Bolívia-Bra...
1          1717813              Gasocidente do Mato Grosso Ltda - GOM
2       3003146349     Transportadora Sulbrasileira de Gás S.A. - TSB
3       3004992714          Nova Transportadora do Sudeste S.A. - NTS
4       3006248349         Transportadora Associada de Gás S.A. - TAG

📄 Primeiras linhas da tabela: Carregador
   codigo_carregador                   nome_carregador
0             371600  COMPANHIA PARAIBANA DE GAS PBGAS
1             535681                          COMPAGAS
2            1645009                             AMBAR
3            2232025            

## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

In [48]:
print("--- Consulta 1: Instalações 'Ponto de Entrega' em MS ---")
query1 = """
SELECT
    codigo_instalacao_gasoduto,
    nome_instalacao_gasoduto,
    tipo_instalacao,
    municipio,
    uf,
    codigo_instalacao_transporte
FROM
    Instalacao_Gasoduto
WHERE
    municipio = 'Macaé';
"""
cursor1 = conn.execute(query1)
results1 = cursor1.fetchall()

if results1:
    # Get column names from the cursor description
    cols1 = [description[0] for description in cursor1.description]
    df1 = pd.DataFrame(results1, columns=cols1)
    print(df1)
else:
    print("Nenhum resultado encontrado para a Consulta 1.")

print("\n" + "="*50 + "\n") # Separator

--- Consulta 1: Instalações 'Ponto de Entrega' em MS ---
   codigo_instalacao_gasoduto                  nome_instalacao_gasoduto   
0                      220981      Interconexão TECAB (TECAB >> GASCAV)  \
1                      220988      Interconexão TECAB (GASCAV >> TECAB)   
2                      222296  Interconexão TECAB (TECAB >> GASDUC III)   
3                      222298  Interconexão TECAB (GASDUC III >> TECAB)   
4                      222300                      UTE Norte Fluminense   
5                      222301                            UTE Mário Lago   
6                      622774  INTERCONEXÃO CABIÚNAS (GASCAV >> GASDUC)   
7                      622776  INTERCONEXÃO CABIÚNAS (GASDUC >> GASCAV)   
8                      622844  INTERCONEXÃO CABIÚNAS (GASDUC >> GASCAV)   
9                      622845  INTERCONEXÃO CABIÚNAS (GASCAV >> GASDUC)   

        tipo_instalacao municipio  uf  codigo_instalacao_transporte  
0  Ponto de Recebimento     Macaé  RJ          

### 6.1.2 Consulta 2

In [49]:
print("--- Consulta 2: Nome e Município de Instalações 'Ponto de Recebimento' no RJ ---")
query2 = """
SELECT
    codigo_carregador,
    nome_carregador
FROM
    Carregador
WHERE
    nome_carregador = 'Petróleo Brasileiro S.A. - PETROBRAS' OR nome_carregador = 'Shell Energy do Brasil Gás Ltda';
"""
cursor2 = conn.execute(query2)
results2 = cursor2.fetchall()

if results2:
    # Get column names from the cursor description
    cols2 = [description[0] for description in cursor2.description]
    df2 = pd.DataFrame(results2, columns=cols2)
    print(df2)
else:
    print("Nenhum resultado encontrado para a Consulta 2.")

# --- 5. Close the database connection ---
conn.close()

--- Consulta 2: Nome e Município de Instalações 'Ponto de Recebimento' no RJ ---
   codigo_carregador                       nome_carregador
0           33000167  Petróleo Brasileiro S.A. - PETROBRAS
1         9600150046       Shell Energy do Brasil Gás Ltda


## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3

In [50]:
# Consulta 3: Listar todos os contratos com seus respectivos dados de medição.
conn = sqlite3.connect('gas_data.db')
query3 = """
SELECT
    C.nome_contrato,
    M.data_medicao,
    M.valor,
    C.codigo_carregador,
    M.codigo_instalacao_transporte
FROM
    Contrato AS C
INNER JOIN
    Medicao AS M ON C.codigo_carregador = M.codigo_carregador;
"""
cursor3 = conn.execute(query3)
results3 = cursor3.fetchall()

if results3:
    cols3 = [description[0] for description in cursor3.description]
    df3 = pd.DataFrame(results3, columns=cols3)
    print(df3)
else:
    print("Nenhum resultado encontrado para a Consulta 3.")

Nenhum resultado encontrado para a Consulta 3.


### 6.2.2 Consulta 4

In [51]:
# Consulta 4: Calcular o valor total medido para cada contrato.
query4 = """
SELECT
    C.nome_contrato,
    SUM(M.valor) AS ValorTotalMedido
FROM
    Contrato AS C
INNER JOIN
    Medicao AS M ON C.codigo_carregador = M.codigo_carregador
GROUP BY
    C.nome_contrato;
"""
cursor4 = conn.execute(query4)
results4 = cursor4.fetchall()

if results4:
    cols4 = [description[0] for description in cursor4.description]
    df4 = pd.DataFrame(results4, columns=cols4)
    print(df4)
else:
    print("Nenhum resultado encontrado para a Consulta 4.")

Nenhum resultado encontrado para a Consulta 4.


### 6.2.3 Consulta 5

In [52]:
# Consulta 5: Encontrar contratos que possuem medições para uma instalação de transporte específica.
query5 = """
SELECT DISTINCT
    C.nome_contrato,
    C.codigo_instalacao_transporte
FROM
    Contrato AS C
INNER JOIN
    Medicao AS M
ON
    C.codigo_carregador = M.codigo_carregador AND C.codigo_instalacao_transporte = M.codigo_instalacao_transporte
WHERE
    C.codigo_instalacao_transporte = '505444'; -- Exemplo para o contrato 'Fronteira (Argentina) - Uruguaiana'
"""
cursor5 = conn.execute(query5)
results5 = cursor5.fetchall()

if results5:
    cols5 = [description[0] for description in cursor5.description]
    df5 = pd.DataFrame(results5, columns=cols5)
    print(df5)
else:
    print("Nenhum resultado encontrado para a Consulta 5.")


Nenhum resultado encontrado para a Consulta 5.


## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6

In [53]:
# Consulta 6: Encontrar carregadores da instalação de gasoduto Corumbá
query6 = """
SELECT DISTINCT CA.codigo_carregador, CA.nome_carregador
FROM
    Contrato AS C
INNER JOIN 
    Instalacao_Transporte as IT
ON
    C.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN
    Instalacao_Gasoduto as IG
ON
    IG.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN
    Carregador as CA
ON
    CA.codigo_carregador = C.codigo_carregador
WHERE
    IG.nome_instalacao_gasoduto = "Corumbá"
"""
cursor6 = conn.execute(query6)
results6 = cursor6.fetchall()

if results6:
    cols6 = [description[0] for description in cursor6.description]
    df6 = pd.DataFrame(results6, columns=cols6)
    print(df6)
else:
    print("Nenhum resultado encontrado para a Consulta 5.")
conn.close()

    codigo_carregador                        nome_carregador
0              535681                               COMPAGAS
1            16974249                                   GALP
2            33000167   Petróleo Brasileiro S.A. - PETROBRAS
3            72300122                                  SCGAS
4            86864543                                 SULGAS
5          3002741679                                  MSGAS
6          3004423567                                  ENEVA
7          3034456148                                   YPFB
8          3056123140                                    MTX
9          7719046324                                   EDGE
10         7748516886  MERCURIO COMERCIALIZADORA DE GAS LTDA
11         9600150046        Shell Energy do Brasil Gás Ltda


### 6.3.2 Consulta 7

### 6.3.3 Consulta 8

## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

### 6.4.2 Consulta 10

# 7. Autoavaliação dos membros